In [35]:
import os
import re
import json
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

import requests
import pandas as pd
from tqdm import tqdm  # console progress bar

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1")

USE_CONTEXT = False          
ACCEPT_THRESHOLD = 0.0       
REQUEST_TIMEOUT_S = 30       
MAX_RETRIES = 2              
HEARTBEAT_EVERY = 25         

def _clean(s: Optional[str]) -> str:
    return (s or "").strip()

def _to_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


In [36]:

def load_data(path: str) -> List[Dict[str, Any]]:
    """
    Load a dataset file that is either:
      - a single JSON object,
      - a list of JSON objects,
      - or a JSONL file (one object per line).
    Returns a list of objects.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    if p.suffix.lower() == ".jsonl":
        rows = []
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows
    else:
        obj = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(obj, list):
            return obj
        return [obj]

def extract_targets(obj: Dict[str, Any]) -> List[str]:
    # "target" is a list of acceptable strings
    return [_clean(t) for t in _to_list(obj.get("target", [])) if _clean(t)]

def extract_candidates(obj: Dict[str, Any]) -> List[str]:
    # "clarified_all_ans" is a List[List[str]]; flatten it
    out: List[str] = []
    blocks = _to_list(obj.get("clarified_all_ans", []))
    for block in blocks:
        for s in _to_list(block):
            s_clean = _clean(s)
            if s_clean:
                out.append(s_clean)
    return out

def to_context(obj: Dict[str, Any]) -> str:
    # Kept for compatibility; NOT used when USE_CONTEXT=False
    return _clean(obj.get("input", ""))


In [37]:

JUDGE_SYSTEM_PROMPT = (
    "You judge whether CANDIDATE is semantically equivalent to TARGET.\n"
    "Focus on meaning, not wording. Ignore extra fluff.\n"
    'Return ONLY JSON: {"equivalent": true|false, "score": 0..1, "rationale": "..."}\n'
    "- TRUE only if a grader would accept CANDIDATE in place of TARGET with no change of meaning.\n"
    "- Minor rephrasing/synonyms → may be TRUE. Broader/narrower/related terms → FALSE.\n"
    "- Score reflects confidence; 1.0 = exact same meaning.\n"
)

def build_judge_prompt(target: str, candidate: str, context: str = "", use_context: bool = False) -> str:
    """
    Builds the user message. If use_context=False, context is ignored.
    """
    if use_context:
        return (
            "CONTEXT:\n{ctx}\n\nTARGET:\n{tgt}\n\nCANDIDATE:\n{cand}\n\n"
            "Decide if CANDIDATE expresses the same meaning as TARGET in this context."
        ).format(ctx=context, tgt=target, cand=candidate)
    else:
        return (
            "TARGET:\n{tgt}\n\nCANDIDATE:\n{cand}\n\n"
            "Decide if CANDIDATE expresses the same meaning as TARGET."
        ).format(tgt=target, cand=candidate)


In [38]:

def _post_chat(model: str, system: str, user: str, host: str, temperature: float = 0.0) -> str:
    """
    Calls Ollama /api/chat and returns the assistant message content as a string.
    Raises on hard errors; caller handles retries.
    """
    url = host.rstrip("/") + "/api/chat"
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                url,
                json={
                    "model": model,
                    "messages": messages,
                    "stream": False,
                    "options": {"temperature": temperature},
                },
                timeout=REQUEST_TIMEOUT_S,
            )
            resp.raise_for_status()
            content = (resp.json().get("message", {}) or {}).get("content", "")
            return _clean(content)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(1.0)

def _parse_json_from_content(content: str) -> Dict[str, Any]:
    """
    Extracts the first {...} block and parses it as JSON. Returns {} on failure.
    """
    m = re.search(r"\{.*\}", content, flags=re.DOTALL)
    s = m.group(0) if m else content
    try:
        return json.loads(s)
    except Exception:
        return {}


In [39]:

def judge_equivalence_once(
    model: str,
    host: str,
    target: str,
    candidate: str,
    context: str = "",
    use_context: bool = USE_CONTEXT,
    temperature: float = 0.0,
) -> Dict[str, Any]:
    """
    Ask the LLM to decide semantic equivalence between TARGET and CANDIDATE.
    If use_context=False (default), the context is ignored.
    Returns dict: {"equivalent": bool, "score": float, "rationale": str, "raw": str}
    """
    user = build_judge_prompt(target=target, candidate=candidate, context=context, use_context=use_context)
    content = _post_chat(model, JUDGE_SYSTEM_PROMPT, user, host, temperature=temperature)
    data = _parse_json_from_content(content)

    eq = bool(data.get("equivalent", False))
    try:
        score = float(data.get("score", 1.0 if eq else 0.0))
    except Exception:
        score = 1.0 if eq else 0.0
    rationale = _clean(data.get("rationale", ""))

    # Optional strictness gate
    if eq and score < ACCEPT_THRESHOLD:
        eq = False

    return {"equivalent": eq, "score": score, "rationale": rationale, "raw": content}


In [40]:

def best_equivalence_against_targets(
    targets: List[str],
    candidate: str,
    model: str = OLLAMA_MODEL,
    host: str = OLLAMA_HOST,
    context: str = "",              # ignored when USE_CONTEXT=False
    use_context: bool = USE_CONTEXT,
) -> Dict[str, Any]:
    """
    Runs judge_equivalence_once for candidate against each target; returns the best one by
    (score, equivalent) descending.
    """
    results = []
    for tgt in targets:
        r = judge_equivalence_once(model, host, tgt, candidate, context=context, use_context=use_context)
        r["target"] = tgt
        results.append(r)

    # sort by score desc; if tie, prefer equivalent=True
    results.sort(key=lambda d: (d.get("score", 0.0), bool(d.get("equivalent", False))), reverse=True)
    best = results[0] if results else {"equivalent": False, "score": 0.0, "rationale": "", "target": "", "raw": ""}

    return {
        "best_equivalent": bool(best.get("equivalent", False)),
        "best_score": float(best.get("score", 0.0)),
        "best_rationale": _clean(best.get("rationale", "")),
        "best_target": _clean(best.get("target", "")),
        "judge_raw": best.get("raw", ""),
    }


In [13]:
def evaluate_file(
    path: str,
    out_csv: Optional[str] = None,
    model: str = OLLAMA_MODEL,
    host: str = OLLAMA_HOST,
    use_context: bool = False,   # keep False to ignore context
) -> pd.DataFrame:
    objs = load_data(path)
    out_rows: List[Dict[str, Any]] = []

    # We progress per candidate
    total_candidates = 0
    for o in objs:
        n = len(extract_candidates(o))
        total_candidates += (n if n > 0 else 1)
    if total_candidates == 0:
        total_candidates = 1

    with tqdm(total=total_candidates, desc="Judging answers", unit="cand") as pbar:
        for i, obj in enumerate(objs):
            uid = obj.get("id", "ex-{0}".format(i))
            context = to_context(obj)             # ignored unless use_context=True
            targets = extract_targets(obj)
            candidates = extract_candidates(obj)

            if not targets or not candidates:
                out_rows.append({
                    "id": uid,
                    "candidate": "",
                    "best_equivalent": False,
                    "best_score": 0.0,
                    "best_rationale": "Missing targets or candidates",
                    "best_target": "",
                    "context": context,
                })
                pbar.update(1)
                continue

            for cand in candidates:
                if pbar.n % HEARTBEAT_EVERY == 0:
                    print("[heartbeat] processed {0} candidates…".format(pbar.n), flush=True)

                best = best_equivalence_against_targets(
                    targets=targets,
                    candidate=cand,
                    model=model,
                    host=host,
                    context=context,
                    use_context=use_context,
                )
                out_rows.append({
                    "id": uid,
                    "candidate": cand,
                    "best_equivalent": best["best_equivalent"],
                    "best_score": best["best_score"],
                    "best_rationale": best["best_rationale"],
                    "best_target": best["best_target"],
                    "context": context,
                })
                pbar.update(1)

    df = pd.DataFrame(out_rows)
    if out_csv:
        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False, encoding="utf-8")
    return df


In [45]:
squadv2 = pd.read_csv('squadv2.csv')
trivqa = pd.read_csv('trivqa.csv')
truthqa = pd.read_csv('truthqa.csv') # selected answer 

In [51]:
d = {'squadv2': squadv2['best_equivalent'].mean(), 'trivqa': trivqa['best_equivalent'].mean(), 'truthqa': truthqa['best_equivalent'].mean()}

In [57]:
pd.DataFrame(d, index  = ['accuracy'])

,squadv2,trivqa,truthqa
accuracY,0.716759,0.443147,0.348731


In [60]:
squadv2.iloc[1]

id                                                              ex-0
candidate          The correct answer is:\n\n**Particular Churche...
best_equivalent                                                 True
best_score                                                       0.9
best_rationale     The candidate provides a clear explanation of ...
best_target                                               full union
context            Context: The Roman Catholic Church canon law a...
Name: 1, dtype: object